In [1]:

# imports
import os
import sys
import types
import json
import base64

# figure size/format
fig_width = 5
fig_height = 4
fig_format = 'png'
fig_dpi = 96
interactivity = ''
is_shiny = False
is_dashboard = False
plotly_connected = True

# matplotlib defaults / format
try:
  import matplotlib.pyplot as plt
  plt.rcParams['figure.figsize'] = (fig_width, fig_height)
  plt.rcParams['figure.dpi'] = fig_dpi
  plt.rcParams['savefig.dpi'] = "figure"

  # IPython 7.14 deprecated set_matplotlib_formats from IPython
  try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
  except ImportError:
    # Fall back to deprecated location for older IPython versions
    from IPython.display import set_matplotlib_formats
    
  set_matplotlib_formats(fig_format)
except Exception:
  pass

# plotly use connected mode
try:
  import plotly.io as pio
  if plotly_connected:
    pio.renderers.default = "notebook_connected"
  else:
    pio.renderers.default = "notebook"
  for template in pio.templates.keys():
    pio.templates[template].layout.margin = dict(t=30,r=0,b=0,l=0)
except Exception:
  pass

# disable itables paging for dashboards
if is_dashboard:
  try:
    from itables import options
    options.dom = 'fiBrtlp'
    options.maxBytes = 1024 * 1024
    options.language = dict(info = "Showing _TOTAL_ entries")
    options.classes = "display nowrap compact"
    options.paging = False
    options.searching = True
    options.ordering = True
    options.info = True
    options.lengthChange = False
    options.autoWidth = False
    options.responsive = True
    options.keys = True
    options.buttons = []
  except Exception:
    pass
  
  try:
    import altair as alt
    # By default, dashboards will have container sized
    # vega visualizations which allows them to flow reasonably
    theme_sentinel = '_quarto-dashboard-internal'
    def make_theme(name):
        nonTheme = alt.themes._plugins[name]    
        def patch_theme(*args, **kwargs):
            existingTheme = nonTheme()
            if 'height' not in existingTheme:
              existingTheme['height'] = 'container'
            if 'width' not in existingTheme:
              existingTheme['width'] = 'container'

            if 'config' not in existingTheme:
              existingTheme['config'] = dict()
            
            # Configure the default font sizes
            title_font_size = 15
            header_font_size = 13
            axis_font_size = 12
            legend_font_size = 12
            mark_font_size = 12
            tooltip = False

            config = existingTheme['config']

            # The Axis
            if 'axis' not in config:
              config['axis'] = dict()
            axis = config['axis']
            if 'labelFontSize' not in axis:
              axis['labelFontSize'] = axis_font_size
            if 'titleFontSize' not in axis:
              axis['titleFontSize'] = axis_font_size  

            # The legend
            if 'legend' not in config:
              config['legend'] = dict()
            legend = config['legend']
            if 'labelFontSize' not in legend:
              legend['labelFontSize'] = legend_font_size
            if 'titleFontSize' not in legend:
              legend['titleFontSize'] = legend_font_size  

            # The header
            if 'header' not in config:
              config['header'] = dict()
            header = config['header']
            if 'labelFontSize' not in header:
              header['labelFontSize'] = header_font_size
            if 'titleFontSize' not in header:
              header['titleFontSize'] = header_font_size    

            # Title
            if 'title' not in config:
              config['title'] = dict()
            title = config['title']
            if 'fontSize' not in title:
              title['fontSize'] = title_font_size

            # Marks
            if 'mark' not in config:
              config['mark'] = dict()
            mark = config['mark']
            if 'fontSize' not in mark:
              mark['fontSize'] = mark_font_size

            # Mark tooltips
            if tooltip and 'tooltip' not in mark:
              mark['tooltip'] = dict(content="encoding")

            return existingTheme
            
        return patch_theme

    # We can only do this once per session
    if theme_sentinel not in alt.themes.names():
      for name in alt.themes.names():
        alt.themes.register(name, make_theme(name))
      
      # register a sentinel theme so we only do this once
      alt.themes.register(theme_sentinel, make_theme('default'))
      alt.themes.enable('default')

  except Exception:
    pass

# enable pandas latex repr when targeting pdfs
try:
  import pandas as pd
  if fig_format == 'pdf':
    pd.set_option('display.latex.repr', True)
except Exception:
  pass

# interactivity
if interactivity:
  from IPython.core.interactiveshell import InteractiveShell
  InteractiveShell.ast_node_interactivity = interactivity

# NOTE: the kernel_deps code is repeated in the cleanup.py file
# (we can't easily share this code b/c of the way it is run).
# If you edit this code also edit the same code in cleanup.py!

# output kernel dependencies
kernel_deps = dict()
for module in list(sys.modules.values()):
  # Some modules play games with sys.modules (e.g. email/__init__.py
  # in the standard library), and occasionally this can cause strange
  # failures in getattr.  Just ignore anything that's not an ordinary
  # module.
  if not isinstance(module, types.ModuleType):
    continue
  path = getattr(module, "__file__", None)
  if not path:
    continue
  if path.endswith(".pyc") or path.endswith(".pyo"):
    path = path[:-1]
  if not os.path.exists(path):
    continue
  kernel_deps[path] = os.stat(path).st_mtime
print(json.dumps(kernel_deps))

# set run_path if requested
run_path = 'QzpcVXNlcnNcbnBhZGFsa2FcRG9jdW1lbnRzXFJlcG9zaXRvcmllc1xBRDY5OC1nZW5lcmF0aXZlLWFpLWZvci1CQQ=='
if run_path:
  # hex-decode the path
  run_path = base64.b64decode(run_path.encode("utf-8")).decode("utf-8")
  os.chdir(run_path)

# reset state
%reset

# shiny
# Checking for shiny by using False directly because we're after the %reset. We don't want
# to set a variable that stays in global scope.
if False:
  try:
    import htmltools as _htmltools
    import ast as _ast

    _htmltools.html_dependency_render_mode = "json"

    # This decorator will be added to all function definitions
    def _display_if_has_repr_html(x):
      try:
        # IPython 7.14 preferred import
        from IPython.display import display, HTML
      except:
        from IPython.core.display import display, HTML

      if hasattr(x, '_repr_html_'):
        display(HTML(x._repr_html_()))
      return x

    # ideally we would undo the call to ast_transformers.append
    # at the end of this block whenver an error occurs, we do 
    # this for now as it will only be a problem if the user 
    # switches from shiny to not-shiny mode (and even then likely
    # won't matter)
    import builtins
    builtins._display_if_has_repr_html = _display_if_has_repr_html

    class _FunctionDefReprHtml(_ast.NodeTransformer):
      def visit_FunctionDef(self, node):
        node.decorator_list.insert(
          0,
          _ast.Name(id="_display_if_has_repr_html", ctx=_ast.Load())
        )
        return node

      def visit_AsyncFunctionDef(self, node):
        node.decorator_list.insert(
          0,
          _ast.Name(id="_display_if_has_repr_html", ctx=_ast.Load())
        )
        return node

    ip = get_ipython()
    ip.ast_transformers.append(_FunctionDefReprHtml())

  except:
    pass

def ojs_define(**kwargs):
  import json
  try:
    # IPython 7.14 preferred import
    from IPython.display import display, HTML
  except:
    from IPython.core.display import display, HTML

  # do some minor magic for convenience when handling pandas
  # dataframes
  def convert(v):
    try:
      import pandas as pd
    except ModuleNotFoundError: # don't do the magic when pandas is not available
      return v
    if type(v) == pd.Series:
      v = pd.DataFrame(v)
    if type(v) == pd.DataFrame:
      j = json.loads(v.T.to_json(orient='split'))
      return dict((k,v) for (k,v) in zip(j["index"], j["data"]))
    else:
      return v

  v = dict(contents=list(dict(name=key, value=convert(value)) for (key, value) in kwargs.items()))
  display(HTML('<script type="ojs-define">' + json.dumps(v) + '</script>'), metadata=dict(ojs_define = True))
globals()["ojs_define"] = ojs_define
globals()["__spec__"] = None

{"C:\\Program Files\\Python312\\Lib\\importlib\\_bootstrap.py": 1744131412.0, "C:\\Program Files\\Python312\\Lib\\importlib\\_bootstrap_external.py": 1744131412.0, "C:\\Program Files\\Python312\\Lib\\zipimport.py": 1744131414.0, "C:\\Program Files\\Python312\\Lib\\codecs.py": 1744131412.0, "C:\\Program Files\\Python312\\Lib\\encodings\\aliases.py": 1744131412.0, "C:\\Program Files\\Python312\\Lib\\encodings\\__init__.py": 1744131412.0, "C:\\Program Files\\Python312\\Lib\\encodings\\utf_8.py": 1744131412.0, "C:\\Program Files\\Python312\\Lib\\encodings\\cp1252.py": 1744131412.0, "C:\\Program Files\\Python312\\Lib\\abc.py": 1744131412.0, "C:\\Program Files\\Python312\\Lib\\io.py": 1744131412.0, "C:\\Program Files\\Python312\\Lib\\stat.py": 1744131412.0, "C:\\Program Files\\Python312\\Lib\\_collections_abc.py": 1744131412.0, "C:\\Program Files\\Python312\\Lib\\genericpath.py": 1744131412.0, "C:\\Program Files\\Python312\\Lib\\ntpath.py": 1744131412.0, "C:\\Program Files\\Python312\\Lib\\o

In [2]:
# | echo: false
# | output: markdown

import pandas as pd
from IPython.display import display, HTML
import sys

# -------------------------------------------------
# Clear module cache (important for Quarto reruns)
# -------------------------------------------------
for m in list(sys.modules):
    if m.startswith(("schedules", "bu_calendar")):
        del sys.modules[m]

from schedules.semester_planner import build_section_schedules

# =================================================
# USER INPUT (only edit this section)
# =================================================
semester = "Summer"
year = 2026
modalities = ["o1"]

start_dates_by_semester = {
    "Spring": {
        "oncampus": "2026-01-26",
        "o2": "2026-01-29",
    },
    "Summer": {
        "oncampus": "2026-05-18",
        "o1": "2026-05-05",
        "o2": "2026-05-18",
    },
    "Fall": {
        "oncampus": "2026-09-14",
        "o1": "2026-09-17",
    },
}

section_plan = build_section_schedules(
    semester=semester,
    year=year,
    modalities=modalities,
    start_dates_by_semester=start_dates_by_semester,
)

weekday_map = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

summary = pd.DataFrame([
    {
        "Section": section,
        "Modality": plan.modality,
        "Class Days": "/".join(weekday_map[d] for d in plan.class_days),
        "Lectures": plan.lecture_count,
        "Start Date": pd.to_datetime(plan.start_date).strftime("%d-%b (%a)"),
        "Module 0 Opens": plan.module0_open.strftime("%d-%b (%a)"),
    }
    for section, plan in section_plan.items()
])

styled = (
    summary
    .style
    .hide(axis="index")
    .set_table_styles([
        {"selector": "th", "props": [("text-align", "left")]},
        {"selector": "td", "props": [
            ("text-align", "left"),
            ("vertical-align", "top"),
            ("white-space", "nowrap")
        ]},
    ])
)

display(HTML(styled.to_html()))

Section,Modality,Class Days,Lectures,Start Date,Module 0 Opens
O1,o1,Tue/Thu,14,05-May (Tue),25-Apr (Sat)


In [3]:
#| echo: false
#| tbl-colwidths: [10,10,15,15]

import pandas as pd
from IPython.display import HTML
from schedules.semester_planner import build_section_schedules
from schedules.deliverables.online_deliverables import apply_online_deliverables
from schedules.presentation import format_deliverables_view

from schedules.config import semester, year, start_dates_by_semester

modalities = ["O1"]

section_plan = build_section_schedules(
    semester=semester,
    year=year,
    modalities=modalities,
    start_dates_by_semester=start_dates_by_semester,
)

frames = []
for section, plan in section_plan.items():
    if plan.modality == "oncampus":
        continue
    schedule_df = pd.DataFrame({
        "Date": [d.strftime("%d-%b (%a)") for d in plan.lecture_dates],
    })
    schedule_df = apply_online_deliverables(
        schedule_df,
        plan.lecture_dates,
    )
    schedule_df = format_deliverables_view(schedule_df)
    schedule_df.insert(0, "Section", section)
    frames.append(schedule_df)

online_df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

HTML(online_df.to_html(index=False))

Section,Date,Assignments,Project Milestones,Participation
O1,05-May (Tue),A1 (Due: May 14),,
O1,07-May (Thu),,,
O1,12-May (Tue),A2 (Due: May 21),,
O1,14-May (Thu),,Ungraded Milestone 1,
O1,19-May (Tue),A3 (Due: May 28),,
O1,21-May (Thu),,Ungraded Milestone 2,
O1,26-May (Tue),A4 (Due: Jun 04),,
O1,28-May (Thu),,,
O1,02-Jun (Tue),,,
O1,04-Jun (Thu),,Ungraded Milestone 3,


In [4]:
#| echo: false
#| tbl-colwidths: [10,10,15,15]
#| eval: false

import pandas as pd
from schedules.deliverables.oncampus_deliverables import apply_oncampus_deliverables
from schedules.presentation import format_deliverables_view
from IPython.display import HTML
import sys

# -------------------------------------------------
# Clear module cache (important for Quarto re-runs)
# -------------------------------------------------
for m in list(sys.modules):
    if m.startswith(("schedules", "bu_calendar", "schedules.deliverables")):
        del sys.modules[m]

from schedules.semester_planner import build_section_schedules

# ==================================================
# USER INPUT
# ==================================================
from schedules.config import semester, year, start_dates_by_semester

modalities = ["oncampus"]

section_plan = build_section_schedules(
    semester=semester,
    year=year,
    modalities=modalities,
    start_dates_by_semester=start_dates_by_semester,
)

frames = []
for section, plan in section_plan.items():
    schedule_df = pd.DataFrame({
        "Date": [d.strftime("%d-%b (%a)") for d in plan.lecture_dates],
    })
    class_day_index = plan.class_days[0]

    schedule_df = apply_oncampus_deliverables(
        schedule_df,
        plan.lecture_dates,
        class_day_index=class_day_index,
    )
    schedule_df = format_deliverables_view(schedule_df)
    schedule_df.insert(0, "Section", section)
    frames.append(schedule_df)

oncampus_df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

output_file = f"./data/{semester.lower()}{str(year)[-2:]}_deliverables_oncampus.xlsx"
oncampus_df.to_excel(output_file, index=False)

HTML(oncampus_df.to_html(index=False))
